In [1]:
import os
from pathlib import Path
from typing import List

# LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    UnstructuredFileLoader
)
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

print("All libraries imported successfully!")

/Users/udaybijjala/miniconda3/envs/myenv_rag/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


All libraries imported successfully!


In [2]:
folder_path = "documents/"
folder = Path(folder_path)
file_stats = {}

if not folder.exists():
    print(f"Folder '{folder_path}' does not exist!")
else:
    print(f"Folder '{folder_path}' exists")


Folder 'documents/' exists


In [3]:
supported_extensions = [".pdf", ".txt", ".docx", ".doc", ".md"]
file_count = 0
documents = []

for file_path in folder.iterdir():
    print(file_path, type(file_path))
    if file_path.is_file() and file_path.suffix.lower() in supported_extensions:
        file_count += 1
        print(f"Loading: {file_path.name}")
        file_extension = Path(file_path).suffix.lower()
    
        if file_extension == ".pdf":
            loader = PyPDFLoader(str(file_path))
        elif file_extension == ".txt":
            loader = TextLoader(str(file_path), encoding="utf-8")
        elif file_extension in [".docx", ".doc"]:
            loader = Docx2txtLoader(str(file_path))
        else:
            loader = UnstructuredFileLoader(str(file_path))


        docs = loader.load()
        # Add document name to metadata
        for doc in docs:
            doc.metadata["document_name"] = file_path.name
        
        documents.extend(docs)
        file_stats[file_path.name] = len(docs)

print(f"\n{'='*50}")
print(f"Files loaded: {file_count}")
print(f"Total pages/documents: {len(documents)}")
print(f"{'='*50}")
print("\nBreakdown:")
for filename, page_count in file_stats.items():
    print(f"  {filename}: {page_count} pages")
print(f"{'='*50}\n")

documents/RFP SM-PRC0001034 Addendum 1.pdf <class 'pathlib.PosixPath'>
Loading: RFP SM-PRC0001034 Addendum 1.pdf
documents/Scope responses for ID Verification.pdf <class 'pathlib.PosixPath'>
Loading: Scope responses for ID Verification.pdf
documents/Alamo College RFP V1.2.pdf <class 'pathlib.PosixPath'>
Loading: Alamo College RFP V1.2.pdf


/Users/udaybijjala/miniconda3/envs/myenv_rag/lib/python3.10/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


documents/RFP_Som-AJKEditsv1.pdf <class 'pathlib.PosixPath'>
Loading: RFP_Som-AJKEditsv1.pdf

Files loaded: 4
Total pages/documents: 41

Breakdown:
  RFP SM-PRC0001034 Addendum 1.pdf: 3 pages
  Scope responses for ID Verification.pdf: 2 pages
  Alamo College RFP V1.2.pdf: 20 pages
  RFP_Som-AJKEditsv1.pdf: 16 pages



In [32]:
chunks

[Document(page_content='REQUEST FOR PROPOSAL   \nRFP # SM-PRC0001034 \nRecord and Review  \nDue:  05.04.2021 before 12:00 p.m. MT \n \n \nADDENDUM 1 \n \n1. Questions and answers attached. \n \n \n \n \n \n \n \n \n \n \n \n\uf076 Please acknowledge by including this signed sheet within your proposal submission . \n \nX              \nSignature of Authorized Agent      Date \n \n              \nPrinted Name of Authorized Agent      Title \n \n              \nFirm Name        Website \n \n( ) -  ( ) -        \nPhone Number   Fax Number    Email Address', metadata={'source': 'documents/RFP SM-PRC0001034 Addendum 1.pdf', 'page': 0, 'document_name': 'RFP SM-PRC0001034 Addendum 1.pdf'}),
 Document(page_content='REQUEST FOR PROPOSAL   \nRFP # SM-PRC0001034 \nRecord and Review  \nDue:  05.04.2021 before 12:00 p.m. MT \n \n \nQuestions & Answers \n \n1.Q. Please provide details on the “Student Support and escalation” function expected to be part of the \nproctoring system   \n \n1.A. The vendo

In [4]:
chunk_size = 5000
chunk_overlap = 1000

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
chunks = text_splitter.split_documents(documents)
print(f"Documents split into {len(chunks)} chunks")

Documents split into 25 chunks


In [ ]:
import shutil

CHROMA_DB_PATH = "./chroma_db_langchain"
COLLECTION_NAME = "document_chunks_langchain"



# Initialize LangChain OpenAI Embeddings (Ada)
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)

vectorstore = None

print("Creating ChromaDB vectorstore with LangChain...")
print(f"Generating embeddings for {len(chunks)} chunks...")

# Create vectorstore from documents (embeddings generated automatically)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DB_PATH
    
)

print(f"Vectorstore created with {vectorstore._collection.count()} chunks!")
print(f"Persisted to: {CHROMA_DB_PATH}")


Creating ChromaDB vectorstore with LangChain...
Generating embeddings for 25 chunks...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vectorstore created with 25 chunks!
Persisted to: ./chroma_db_langchain


In [18]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever created (top 5 similar chunks)")

Retriever created (top 5 similar chunks)


In [19]:
retriever.invoke("what is the main of the topic of the documents?")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(page_content='ExamRoom.AI  Page  | 5  \n \n \nprocessing times, ensure compliance with state -specific requirements, and improve the \noverall user experience for both staff and applicants.  \nc. Describe  how your solution  will be developed,  specifying  the overall  technology  and \napplication architecture. Include details on the following aspects:  \n \n1) The core technologies,  frameworks,  and programming  languages  that will be used.  \nAnswer:  \nCore Technologies - Open CV and NLP & LLM  \nFramework – LangChain, LamaIndex, Hugging Face agents  \nProgramming Language – Python  \n \n2) The architecture  design  (e.g.,  microservices,  monolithic,  cloud -based).  \nAnswer:  \nAI based: Cloud based  \nApplication: microservice  \n \n3) Any third-party  or off-the-shelf  vendor  solutions  that will be integrated,  and their \nspecific roles within the system.  \nAnswer:  \nNo \n \nd. Describe  the hosting  requirements  and security  measures  for the solution.  \nA

In [29]:
OPENAI_API_KEY = "API_KEY"  # Replace with your actual API key
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7,
    max_tokens=1000 
)

resp = llm.invoke("what is llm, explain about cricket, tell me a joke, and also tell me about the documents I have uploaded?")

In [30]:
resp

AIMessage(content="LLM stands for Master of Laws, which is a postgraduate degree in law that is typically pursued by students who have already completed their undergraduate studies in law. It is a specialized degree that allows students to focus on a specific area of law, such as international law, human rights law, or intellectual property law.\n\nCricket is a popular sport that is played between two teams of eleven players each. The game involves hitting a ball with a bat and scoring runs by running between two sets of wickets. The team with the most runs at the end of the game wins. Cricket is most popular in countries such as India, England, Australia, and South Africa.\n\nHere's a joke for you: Why did the scarecrow win an award? Because he was outstanding in his field!\n\nAs an AI assistant, I do not have the ability to view or access any documents that you may have uploaded. If you have any specific questions or need assistance with the documents, please provide more details or 

In [ ]:
template = """You are a helpful assistant that answers questions based on the provided context.
Always base your answers on the given context. If the context doesn't contain enough information
to answer the question, say so. When possible, cite the source document for your information.

Context:
{context}

Question: {question}

Please provide a comprehensive answer based on the context above. Cite the source documents when relevant."""

prompt = ChatPromptTemplate.from_template(template)

In [27]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are a helpful assistant that answers questions based on the provided context.\nAlways base your answers on the given context. If the context doesn't contain enough information\nto answer the question, say so. When possible, cite the source document for your information.\n\nContext:\n{context}\n\nQuestion: {question}\n\nPlease provide a comprehensive answer based on the context above. Cite the source documents when relevant."))])

In [25]:
def format_docs(docs):
    """Format retrieved documents with source information."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("document_name", "unknown")
        formatted.append(f"[Source {i}: {source}]\n{doc.page_content}")
    return "\n\n".join(formatted)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [31]:
rag_chain.invoke("what is cricket?")

'The context provided is related to an RFP (Request for Proposal) for the purchase of a credit mobility solution by Alamo College. The RFP mentions ExamRoom.AI Corp as the company submitting the proposal. ExamRoom.AI describes their deployment steps for the solution, including sprint planning, customization, testing, deployment, and post-deployment support. They also mention data protection measures, compliance with industry standards, and cloud-native integration capabilities.\n\nHowever, the context does not provide any information related to the sport of cricket. As such, based on the given context, there is no relevant information to provide a comprehensive answer about cricket. If you have any other questions related to the context provided, feel free to ask.'

In [40]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")     # older, cheaper

In [16]:
resp_embed= embeddings.embed_query("What is the capital of France?")

In [17]:
len(resp_embed)

1536

In [34]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {"question": "What is our return policy?", "answer": "Products can be returned within 30 days."},
    {"question": "Do you ship internationally?", "answer": "We ship to 50 countries."},
]

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=PromptTemplate.from_template("Q: {question}\nA: {answer}"),
    prefix="Answer questions about our company:\n",
    suffix="Q: {question}\nA:",
    input_variables=["question"]
)

In [38]:
print(prompt.invoke("What is your customer support email?").text)

Answer questions about our company:


Q: What is our return policy?
A: Products can be returned within 30 days.

Q: Do you ship internationally?
A: We ship to 50 countries.

Q: What is your customer support email?
A:


In [41]:
print(prompt.invoke("What is your customer support email?").text)

Answer questions about our company:


Q: What is our return policy?
A: Products can be returned within 30 days.

Q: Do you ship internationally?
A: We ship to 50 countries.

Q: What is your customer support email?
A:


In [51]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=1,
    max_tokens=10 
)
llm.invoke("what is llm?")

AIMessage(content='LLM stands for Master of Laws, which is', response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 12, 'total_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'length', 'logprobs': None}, id='run-7d047609-540e-41fa-9ccb-3ccfb2b84e2f-0')